# 10. 综合任务

AGDR VLA 入门教程 · 第 3 章 · 综合任务

- **学习目标**：用后半周的方法，借助 Agent 从零开发一个在终端里运行的 TUI Agent，把前半周的原理（Agent Loop、工具、上下文压缩、权限）落到自己的项目里；严格按「理解 → 计划 → 小步修改 → 审查 diff → 验证 → 提交」的流程工作，按功能逐个提交；通过本章的选择题测验
- **前置知识**：作业一学完 02 节即可开始，各功能对应的小节见下文；作业二需要第 01–09 节全部内容；第 1 章的 Git 与 GitHub 操作
- **预计学习时长**：与 01–09 节并行，贯穿全章两周

本节有三项作业：

| 作业 | 内容 | 提交方式 |
| --- | --- | --- |
| 作业一 | 借助 Agent 开发一个 TUI Agent | 在自建仓库中按功能提交；仓库链接通过 Pull Request 提交 |
| 作业二 | Agent 选择题 | 结论写进作业三的 Issue |
| 作业三 | 提交 Issue | Issue |

## 本章回顾

| 小节 | 一句话 |
| --- | --- |
| 01 API 与消息格式 | 模型是无状态的；两种格式结构不同、流程相同；工具由你的程序执行 |
| 02 Agent Loop | Agent 就是「调用模型 → 执行工具 → 回传结果」的循环，由模型决定何时停 |
| 03 上下文压缩 | 历史越来越长；截断、清除旧结果、摘要压缩各有代价；写进文件的不会丢 |
| 04 MCP | 一次实现、处处可用的工具协议；host、client、server；stdout 只能走协议 |
| 05 Skill 与 Hook | Skill 按需加载知识；Hook 在固定时机强制执行 |
| 06 提示词 | 背景、目标、约束、验收标准；先计划后执行；报错贴完整 |
| 07 项目说明文件 | 每次都加载进系统提示词；写具体可执行、带原因的规则 |
| 08 写代码与排错 | 看 diff 而不是看汇报；验证、小步提交，提交由你来做 |
| 09 安全 | 最小权限、执行前确认；prompt injection；保护密钥；核对引用 |

## 作业一：借助 Agent 开发一个 TUI Agent

前半周你用几十行代码写出了 mini agent，知道了 Claude Code、Codex 这类工具在背后做了什么。这次反过来：**借助 Agent，开发一个你自己的终端 Agent**。

这项作业**允许、也要求你用 Agent 写代码**，但这不是「拿 AI 写作业」。它考察的不是代码本身，而是你能不能**主导**一次和 Agent 的合作：定需求、批准计划、审查每一处修改、验证结果、完成提交（08 节）。代码可以交给 Agent 写，**审核和提交必须由你亲自完成**：

- 不能让 Agent 审查它自己的修改。可以让它解释改了什么，但每一处 diff 是否采纳，由你自己读、自己判断
- 不能让 Agent 执行 `git commit`、`git push`，commit message 也由你自己写
- 不能跳过验证，只凭 Agent 的汇报就提交

Shiro 会逐个审查你仓库里的 commit，从提交历史里看你是否真的按流程在工作，也可能请你解释其中任意一个 commit。

为了不和本仓库学习者模式的规则冲突，这项作业在**你自己新建的仓库**里完成，不要在本仓库里开发。

这项作业和小节**并行**：学完 02 节就可以建好仓库、开始开发。之后对照下面表格的「对应小节」一列，学到哪一节，就把对应的功能做进去；第 2 周完成剩下的功能。

### 功能要求

你的 Agent 必须具备下面**全部**功能：

| 功能 | 要求 | 对应小节 |
| --- | --- | --- |
| API 配置 | 能配置 base URL、API key 和模型名（配置文件或环境变量均可）；key 不能写死在代码里，也不能提交进仓库；提供一份不含真实 key 的示例配置 | 01、09 |
| TUI 界面 | 在终端里运行，支持连续多轮对话；用户输入、模型回复、工具调用在界面上明显区分 | 02 |
| Agent Loop | 「调用模型 → 执行工具 → 回传结果」的循环，由模型决定何时结束；设置最大轮数，防止死循环 | 02 |
| 工具 | 至少包括**运行命令、读取文件、修改文件**（整体写入或按内容替换）；可以再加列目录、搜索等 | 02、08 |
| 显式展示工具调用 | 每次调用都在界面上显示：调用了哪个工具、参数是什么、返回了什么（过长的输出可以截断）；修改文件时显示 diff | 02、08 |
| 沙箱 | 启动时指定一个工作目录，文件工具只能访问这个目录内的路径（`../`、绝对路径、符号链接越界都要拒绝）；命令在工作目录下运行，并设置超时 | 02、09 |
| 危险权限审核 | 运行命令、修改文件之前，先展示要做什么，征得用户确认；能识别危险操作（如 `rm -rf`、`sudo`、`git push`、读取 `.env` 或 `~/.ssh`），给出醒目警告或直接拒绝；用户拒绝时，把「被拒绝」作为工具结果告诉模型，让它继续工作 | 09 |
| 上下文压缩 | 对话过长时（例如估算的 token 数或消息条数超过阈值）自动压缩，也提供手动压缩的命令；压缩后消息格式仍然合法，关键信息不丢 | 03 |

技术选择由你决定：

- 语言不限，推荐 Python
- 消息格式用 OpenAI 或 Anthropic 都可以（01 节），在 README 里写明
- TUI 可以用 `textual`、`rich`、`prompt_toolkit` 等库，也可以只用标准库；用了第三方库，要在 `requirements.txt` 或 `pyproject.toml` 里写明依赖
- 可以参考本章的 `mini_agent.py`，但不要原样复制进你的仓库

### 开发流程

**严格按照本章讲的流程开发。** 这才是这项作业真正要考察的内容，功能本身反而是次要的。

**1. 新建仓库**（第 1 章）：在 GitHub 上新建一个**公开**仓库（例如 `tui-agent`），克隆到本地。不要放在本仓库的目录里面。

**2. 自己写 `AGENTS.md`**（07 节），不要让 Agent 写。至少包括：

- 项目是什么，用了哪些技术
- 怎样运行、怎样测试
- 约束：key 不能写进代码；只做本次要求的改动；不要自行提交和推送。每条规则写明原因

如果你用 Claude Code，再建一个只有一行 `@AGENTS.md` 的 `CLAUDE.md`。再写好 `.gitignore`，把存放 key 的配置文件排除在外。这些作为**第一个 commit**。

**3. 设置 Agent 的权限**（09 节）：修改文件、运行命令都要先问你；不允许它提交和推送。

**4. 一个功能一个功能地开发。** 每个功能都完整走一遍 08 节的流程：

| 步骤 | 你要做的 |
| --- | --- |
| 理解 | 让 Agent 先读现有代码，说明它打算怎么做 |
| 计划 | 用 06 节的方法写提示词：背景、目标、约束、验收标准；让它先出计划，你确认后再动手 |
| 小步修改 | 一次只做这一个功能；每次批准前，看清它要运行的命令、要改的内容 |
| 审查 diff | 用 `git diff` 逐处检查，没有要求的改动让它撤回 |
| 验证 | 自己运行一遍，确认功能真的可用；能写测试的最好写一个 |
| 提交 | 由**你自己**执行 `git commit`，自己写 commit message，说明这个功能是什么 |

建议的提交顺序（可以调整，但每个 commit 只做一件事）：

1. `AGENTS.md`、`.gitignore`、README 骨架
2. API 配置与示例配置文件
3. 调用一次模型并显示回复
4. TUI 界面与多轮对话
5. Agent Loop 与读取文件工具，界面显示工具调用过程
6. 沙箱：限制工作目录
7. 运行命令工具与执行前确认
8. 修改文件工具与 diff 展示
9. 危险操作识别
10. 上下文压缩
11. 完善 README：安装、配置、运行方式和功能演示

功能出了 bug，修复也单独提交，message 写明修了什么。**不要**把所有改动攒成一个大 commit，也**不要**在提交作业之后用 rebase、force push 改写已有的提交历史。

**5. 保护 API key**（09 节）：每次提交前，用 `git status`、`git diff --staged` 确认没有带上 key。万一 key 被推送到了 GitHub，删掉文件是不够的，它还留在历史里。要**立刻去平台作废这个 key**，再重新生成一个。

> 开发中遇到概念上的问题，可以回到本仓库根目录启动 Agent，用问答模式提问。

### 提交作业一

开发完成后，把仓库链接通过 Pull Request 提交到本仓库：

1. 同步你的 Fork，新建分支：`homework/tui-agent-你的GitHub用户名`
2. 新建文件 `3. Agent以及AI工具的使用/submission/你的GitHub用户名.md`，按下面的模板填写：

   ```markdown
   # 第 3 章作业一

   - 仓库链接：https://github.com/你的GitHub用户名/仓库名
   - 使用的 Agent 工具和模型：
   - 消息格式：OpenAI / Anthropic
   ```

3. 用 `git status` 确认只添加了这一个文件，然后提交、推送，向原仓库发起 Pull Request，标题格式：`[第3章-作业一] 你的名字`

> 本仓库里只放链接，不要把你的代码复制进来。尤其不要把你仓库里的 `AGENTS.md` 放进本仓库：回想 07 节，Agent 会自动读取工作目录里的 `AGENTS.md`，它可能被当成指令读进去。

**检查标准**（Shiro 会克隆你的仓库，逐个查看 commit）

- 按 README 能安装、配置并运行，上面表格里的功能全部可用
- 工具调用过程在界面上清楚可见；越界的路径被拒绝；危险命令有警告或被拒绝，拒绝后 Agent 能继续工作
- 提交历史按功能拆分，每个 commit 只做一件事，message 清楚；第一个 commit 里有你自己写的 `AGENTS.md`
- 任何一个 commit 里都没有真实的 API key
- 你能解释每一个 commit 改了什么、为什么这样改

## 作业二：Agent 选择题

本章选择题统一在这里做，范围是第 01–09 节的全部内容。

1. 在仓库**根目录**启动 Agent，直接开始对话
2. 选择「答题模式」和「第 3 章」，按提示作答，直到 Agent 宣布完成
3. 完成后，Agent 会给出一份总结，把它复制到作业三的 Issue 里

要求：

- 自己作答，不要让 Agent 直接给出答案
- 答错的题目要弄懂原因，可以切换到问答模式提问，再回来继续
- 如果 Agent 出的题有明显错误，记录下来，写进作业三的 Issue

学完本章，你应该能理解：此刻为你出题的这个 Agent，正是按照根目录的 `AGENTS.md` 和本章的 `Agent选择题协议.md` 在工作。

## 作业三：提交 Issue

在原仓库新建一个 Issue：

**标题**：`[第3章-作业三] 你的名字`

**正文**：

1. **环境**：你用的是哪个 Agent 工具；学习本章时是否配置了 API（在线模式还是离线模式）
2. **作业二的最终结论**与最后一组题目（Agent 给出的总结）
3. **作业一中印象最深的一件事**：Agent 做得特别好的一步，或者你在审查 diff 时发现并纠正的一个问题
4. **一个问题或建议**：本章中你遇到的问题、不清楚的地方，或者对教程的改进建议
5. 如果发现 Agent 出题或讲解有错误，写出具体题目和你的判断

> 粘贴前检查一下，内容里不要包含 API key、密码或私钥。

## 完成检查表

**知识**

- [ ] 前半周：01 API 与消息格式、02 Agent Loop、03 上下文压缩、04 MCP、05 Skill 与 Hook
- [ ] 后半周：06 提示词、07 项目说明文件、08 写代码与排错、09 安全

**作业**

- [ ] 作业一：TUI Agent 的功能全部可用，提交历史按功能拆分，没有泄露 key
- [ ] 作业一：放有仓库链接的 Pull Request 已提交
- [ ] 作业二：Agent 判定完成，结论已记录
- [ ] 作业三：Issue 已提交

全部完成后，就可以进入第 4 章了。从下一章开始，你会在学习 Python 环境、机器学习的过程中真正地用上 Agent。记住本章最重要的一句话：**Agent 是工具，不是替代品。**